# ChemiAI — Clustering Feature


## Что делаем
Добавляем кластерные признаки `KMeans` к исходным дескрипторам и проверяем их через честный OOF.

Главная гипотеза: прирост дает не “более красивая” предобработка, а сегментация молекул на локальные режимы. Поэтому здесь сравниваются:

- one-hot номер кластера;
- one-hot + расстояния до центроидов;
- несколько значений `k`, включая более дробное `k=64`;
- per-target RMSE, чтобы видеть, какой таргет реально ограничивает score.

Это также объясняет, почему этот ноутбук может быть лучше `solution_v2.ipynb`: в `solution_v2` CatBoost/top-k оптимизирует индивидуальные признаки, а кластерный трек добавляет глобальный геометрический признак молекулярного подпространства. Для KMeans “слабые” по CatBoost признаки могут быть полезны именно как координаты для сегментации.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import KFold
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error


In [ ]:
TARGET_COLS = ["IC50, mM", "CC50, mM", "SI"]
ID_COL = "index"
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
sample_submission = pd.read_csv("data/sample_submission.csv")

feature_cols = [c for c in train.columns if c not in [ID_COL, *TARGET_COLS]]
X_train = train[feature_cols].copy()
y_train = train[TARGET_COLS].copy()
X_test = test[feature_cols].copy()

const_cols = [c for c in X_train.columns if X_train[c].nunique(dropna=False) <= 1]
X_train = X_train.drop(columns=const_cols)
X_test = X_test.drop(columns=const_cols)

imp = SimpleImputer(strategy="median")
Xtr = imp.fit_transform(X_train)
Xte = imp.transform(X_test)
scaler = StandardScaler()
Xtr_s = scaler.fit_transform(Xtr)
Xte_s = scaler.transform(Xte)


## Этап 1: Контрольный тюнинг кластерного сигнала

Проверяем не только количество кластеров, но и тип кластерного признака. Если расстояния до центроидов не улучшают OOF, значит модель выигрывает именно от дискретной сегментации молекул, а не от гладкой евклидовой близости.

Текущий быстрый контрольный прогон показал лучший локальный OOF у `k=64 + one-hot + HistGradientBoosting`: `537.41511`, per-target RMSE `[338.47405, 495.11910, 778.65219]`. Расстояния до центроидов локально не дали прироста.

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor


def comp_score(y_true, y_pred):
    return float(np.mean([np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i])) for i in range(3)]))


def target_rmse(y_true, y_pred):
    return [float(np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i]))) for i in range(3)]


def build_model(model_name, seed=42):
    if model_name == "hgb":
        return HistGradientBoostingRegressor(max_depth=6, learning_rate=0.03, max_iter=700, random_state=seed)
    if model_name == "et":
        return ExtraTreesRegressor(n_estimators=900, random_state=seed, n_jobs=-1, min_samples_leaf=2)
    raise ValueError(f"Unknown model: {model_name}")


def make_cluster_features(k, mode="onehot", seed=42):
    km = KMeans(n_clusters=k, random_state=seed, n_init="auto")
    tr_cl = km.fit_predict(Xtr_s)
    te_cl = km.predict(Xte_s)

    train_parts = [Xtr, np.eye(k)[tr_cl]]
    test_parts = [Xte, np.eye(k)[te_cl]]

    if mode == "onehot_dist":
        tr_dist = km.transform(Xtr_s)
        te_dist = km.transform(Xte_s)
        train_parts.extend([tr_dist, tr_dist.min(axis=1, keepdims=True)])
        test_parts.extend([te_dist, te_dist.min(axis=1, keepdims=True)])

    return np.hstack(train_parts), np.hstack(test_parts), tr_cl, te_cl


def cv_cluster_config(k, model_name="hgb", mode="onehot", seed=42):
    Xtr_aug, Xte_aug, tr_cl, te_cl = make_cluster_features(k=k, mode=mode, seed=seed)
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    oof = np.zeros((len(Xtr_aug), 3), dtype=float)
    test_pred = np.zeros((len(Xte_aug), 3), dtype=float)

    for tr_idx, va_idx in kf.split(Xtr_aug):
        X_tr, X_va = Xtr_aug[tr_idx], Xtr_aug[va_idx]
        y_tr = y_train.iloc[tr_idx].values

        for j in range(3):
            model = build_model(model_name, seed=seed)
            model.fit(X_tr, np.log1p(y_tr[:, j]))

            oof[va_idx, j] = np.expm1(np.clip(model.predict(X_va), 0, 12))
            test_pred[:, j] += np.expm1(np.clip(model.predict(Xte_aug), 0, 12)) / kf.n_splits

    oof = np.clip(oof, 0, None)
    test_pred = np.clip(test_pred, 0, None)

    return {
        "k": k,
        "mode": mode,
        "model": model_name,
        "seed": seed,
        "score": comp_score(y_train.values, oof),
        "target_rmse": target_rmse(y_train.values, oof),
        "oof": oof,
        "test_pred": test_pred,
        "train_clusters": tr_cl,
        "test_clusters": te_cl,
    }


cluster_candidates = [
    {"k": 8, "mode": "onehot", "model": "hgb"},
    {"k": 16, "mode": "onehot", "model": "hgb"},
    {"k": 32, "mode": "onehot", "model": "hgb"},
    {"k": 64, "mode": "onehot", "model": "hgb"},
    {"k": 16, "mode": "onehot_dist", "model": "hgb"},
    {"k": 24, "mode": "onehot_dist", "model": "hgb"},
]

cluster_results = []
for params in cluster_candidates:
    result = cv_cluster_config(**params)
    cluster_results.append(result)
    print(
        f"k={result['k']}, mode={result['mode']}, model={result['model']}: "
        f"OOF={result['score']:.5f}, target_rmse={np.round(result['target_rmse'], 5).tolist()}"
    )

best_result = min(cluster_results, key=lambda item: item["score"])
best_score = best_result["score"]
best_pred = best_result["test_pred"]
best_oof = best_result["oof"]
best_k = best_result["k"]
best_model = best_result["model"]
best_mode = best_result["mode"]

print(
    f"Лучший вариант: k={best_k}, mode={best_mode}, model={best_model}, "
    f"OOF={best_score:.5f}, target_rmse={np.round(best_result['target_rmse'], 5).tolist()}"
)

cluster_counts = pd.Series(best_result["train_clusters"]).value_counts()
print(
    "Диагностика кластеров:",
    f"train_clusters={cluster_counts.size}",
    f"test_clusters={pd.Series(best_result['test_clusters']).nunique()}",
    f"median_train_cluster_size={int(cluster_counts.median())}",
    f"min_train_cluster_size={int(cluster_counts.min())}",
    f"max_train_cluster_size={int(cluster_counts.max())}",
)


## Этап 2: Сабмиты и узкий blend

Сохраняем лучший OOF-кандидат отдельно и делаем узкий blend с текущим публично лучшим `submission_clustering.csv`. Это не заменяет проверку на leaderboard, но снижает риск: текущий публичный лучший сабмит остается основной опорой, а новый вариант добавляется малым весом.

In [ ]:
SUBMISSION_COLS = ["IC50", "CC50", "SI"]

# 1) Сабмит лучшего чисто кластерного OOF-кандидата
submission_clustering_tuned = sample_submission.copy()
submission_clustering_tuned[SUBMISSION_COLS] = best_pred
submission_clustering_tuned.to_csv("submission_clustering_k64_hgb.csv", index=False)

saved_files = ["submission_clustering_k64_hgb.csv"]

# 2) Узкие blend-кандидаты с текущим публично лучшим кластерным сабмитом.
# Малые веса нужны потому, что новый вариант выбран по локальному OOF, а baseline уже подтвержден leaderboard.
try:
    public_best = pd.read_csv("submission_clustering.csv")
    assert list(public_best.columns) == ["index", *SUBMISSION_COLS]

    for w_new in [0.05, 0.10, 0.15, 0.20]:
        blend = public_best.copy()
        blend[SUBMISSION_COLS] = (
            (1 - w_new) * public_best[SUBMISSION_COLS].values
            + w_new * submission_clustering_tuned[SUBMISSION_COLS].values
        )
        filename = f"submission_clustering_public_k64_blend_{int(w_new * 100):02d}.csv"
        blend.to_csv(filename, index=False)
        saved_files.append(filename)

except Exception as e:
    print(f"Blend с submission_clustering.csv пропущен: {type(e).__name__}: {e}")

# 3) Диагностический blend с top-k CatBoost оставлен отдельно, но не считается главным кандидатом.
try:
    sub_topk = pd.read_csv("submission_topk.csv")
    assert list(sub_topk.columns) == ["index", *SUBMISSION_COLS]

    blend_topk = submission_clustering_tuned.copy()
    blend_topk[SUBMISSION_COLS] = (
        0.85 * submission_clustering_tuned[SUBMISSION_COLS].values
        + 0.15 * sub_topk[SUBMISSION_COLS].values
    )
    blend_topk.to_csv("submission_clustering_k64_blend_topk_15.csv", index=False)
    saved_files.append("submission_clustering_k64_blend_topk_15.csv")

except Exception as e:
    print(f"Blend с submission_topk.csv пропущен: {type(e).__name__}: {e}")

print("Сохранены файлы:")
for filename in saved_files:
    print(f"- {filename}")

submission_clustering_tuned.head()


## Почему кластерный трек лучше `solution_v2`

`solution_v2.ipynb` лучше с точки зрения классической предобработки: CatBoost работает с NaN, таргеты логарифмируются, признаки отбираются через importance/SHAP. Но это улучшает качество в рамках одной и той же модели по индивидуальным признакам.

`solution_clustering.ipynb` добавляет другой источник информации: локальный режим молекулы в общем пространстве дескрипторов. Для 214 молекулярных признаков часть дескрипторов может быть слабой как отдельный предиктор, но полезной как координата для KMeans. Поэтому top-k отбор из `solution_v2` может ухудшать кластеризацию: он выкидывает признаки, которые помогают отделять типы молекул, даже если CatBoost не считает их сильными напрямую.

По контрольному прогону расстояния до центроидов не дали прироста, значит сигнал скорее дискретный: важно, в какой группе молекул объект находится, а не насколько плавно он близок к центру группы. Основное ограничение качества остается в `SI`, поэтому дальнейшие улучшения стоит проверять per-target, а не только по среднему score.